# Prepration

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [2]:
%matplotlib inline

In [3]:
working_dir = os.getcwd()
FOLDER_NAME = os.path.basename(working_dir)
FILE_NAME = 'cnn_v1'

STOCK = 'BTC'
FREQ = '1h'

TRADE_PRICE = 'close'


SEQ_LENGTH = 48
BATCH_SIZE = 8

TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.1
TEST_SPLIT = 1 - (TRAIN_SPLIT+VAL_SPLIT)

In [4]:
os.chdir(os.path.join('..','data'))
df = pd.read_csv(f'{STOCK}USD_2023_2024.csv', index_col='Gmt time')
df.index = pd.to_datetime(df.index, format="%d.%m.%Y %H:%M:%S.%f", errors='coerce')
df = df.rename(columns={'Date':'time', 'Open': 'open', 'High': 'high', 'Low': 'low', 'Close':'close', 'Volume':'volume'})

In [5]:
os.chdir('..')
from utils.utils import CreateTimeFrames
df = CreateTimeFrames(df, [FREQ])[FREQ]
df = df.iloc[-4050:,:]

In [ ]:
close = df[f'{TRADE_PRICE}'].to_numpy(dtype=np.int32)
print(f'There are {close.size} {TRADE_PRICE} prices.')

In [7]:
indicators = df.copy()

In [ ]:
indicators.columns

# Add indicators
## 1) RSI

In [ ]:
price_kinds = ['close', 'open', 'low', 'high']

In [10]:
import pandas_ta as ta
for _, price in enumerate(price_kinds):
    indicators[f'fe_rsi_{price}'] = ta.rsi(df[f'{price}'], length=14)


## 2) MACD

In [12]:
# best_macd_coefficients = np.load(f'macdxlstm/{SEQ_LENGTH}/best_macd_coefficients.npy')

In [12]:
indicators['fe_macd_opt_close'] = np.load(f'macdxlstm/{SEQ_LENGTH}/macd_line_{SEQ_LENGTH}.npy')
indicators['fe_macd_signal_opt_close'] = np.load(f'macdxlstm/{SEQ_LENGTH}/signal_line_{SEQ_LENGTH}.npy')
indicators['fe_macd_histogram_close'] = np.load(f'macdxlstm/{SEQ_LENGTH}/histogram_{SEQ_LENGTH}.npy')


In [14]:
def calculate_ema(values, period):
    return pd.Series(values).ewm(span=period, adjust=False).mean().values

def calculate_macd(sequence, fast_period, slow_period, signal_period):
    # calculate macd things for a sequence.
    macd_line = calculate_ema(sequence, fast_period)-calculate_ema(sequence, slow_period)
    signal_line = calculate_ema(macd_line, signal_period)
    histogram = macd_line-signal_line
    return macd_line, signal_line, histogram

# macd_lines = []
# signal_lines = []
# histograms = []

In [ ]:
# def create_sequences(data, seq_length):
#     sequences = []
#     for i in range(len(data) - seq_length + 1):
#         sequences.append(data[i:i + seq_length])
#     return np.array(sequences, dtype=np.int64)

# sequences = create_sequences(df[f'{TRADE_PRICE}'].values, SEQ_LENGTH)
# print(f'Created {len(sequences)} sequences of length {SEQ_LENGTH}.')

In [15]:
# for i, seq in enumerate(sequences):
#     f,s,h = best_macd_coefficients[i]
#     macd_line, signal_line, histogram = calculate_macd(seq, fast_period=f,slow_period=s,signal_period=h)
#     macd_lines.append(macd_line)
#     signal_lines.append(signal_line)
#     # histograms.append(histogram)

# macd_lines = np.array(macd_lines)
# signal_lines = np.array(signal_lines)
# # histograms = np.array(histograms)

In [ ]:
# def catch_the_best(df, sequences, seq_len=SEQ_LENGTH):
#     the_best_values = np.zeros(len(df))
#     s = len(df)-seq_len + 1
#     for i in range(s):
#         the_best_values[i] = sequences[i,-1]


#     the_best_values[s-1:] = sequences[-1,:]
#         # if i == s:
#         #     the_best_values[i:] = sequences[i,:]
#         # else:
#         #     the_best_values[i] = sequences[i,-1]

#     return the_best_values

# indicators['fe_macd_opt_close'] = catch_the_best(df, macd_lines)
# indicators['fe_macd_signal_opt_close'] = catch_the_best(df, signal_lines)

In [21]:
# del best_macd_coefficients, close, f, h, i, s, macd_lines, seq, sequences, signal_lines

In [ ]:
indicators.columns

### Traditional MACD

In [ ]:
for _, price in enumerate(price_kinds):
    indicators[f'fe_macd_{price}'] = ta.trend.MACD(indicators[f'{price}'], window_slow=26, window_fast=12, window_sign=9).macd().to_numpy(dtype=np.float32)
    indicators[f'fe_macd_signal_{price}'] = ta.trend.MACD(indicators[f'{price}'], window_slow=26, window_fast=12, window_sign=9).macd_signal().to_numpy(dtype=np.float32)
    indicators[f'fe_macd_histogram_{price}'] = ta.trend.MACD(indicators[f'{price}'], window_slow=26, window_fast=12, window_sign=9).macd_diff().to_numpy(dtype=np.float32)

## 3) Bolinger Bands

In [15]:
for _, price in enumerate(price_kinds):
    a = ta.bbands(df[f'{price}'], length=20, std = 2)
    indicators[f'fe_bb_upper_{price}'] = a['BBU_20_2.0']
    indicators[f'fe_bb_lower_{price}'] = a['BBL_20_2.0']
    indicators[f'fe_bb_mid_{price}'] = a['BBM_20_2.0']
del a

## 4) On balance volume

In [16]:
indicators['fe_obv_close'] = ta.obv(df['close'], df['volume'])

## 5) Accumulation Distribution Indicator(ADI)

In [17]:
indicators['fe_adi'] = ta.ad(df['high'], df['low'], df['close'], df['volume'])

## 6) Average Directional Index(ADX)

In [40]:
import ta
indicators['fe_adx'] = ta.trend.ADXIndicator(indicators['high'], indicators['low'], indicators['close'], window=14).adx().to_numpy(np.float32)

## 7) Aroon Indicator

In [41]:
aroon_indicator = ta.trend.AroonIndicator(high=df['high'], low=df['low'], window=14)
indicators['fe_aroon_up'] = aroon_indicator.aroon_up().to_numpy(dtype=np.float32)
indicators['fe_aroon_down'] = aroon_indicator.aroon_down().to_numpy(dtype=np.float32)

## 8) Stochastic Oscillator

In [44]:
stoch_indicator = ta.momentum.StochasticOscillator(high=df['high'], low=df['low'], close=df['close'], window=14, smooth_window=3)
indicators['fe_so_k'] = stoch_indicator.stoch().to_numpy(dtype=np.float32)
indicators['fe_so_d'] = stoch_indicator.stoch_signal().to_numpy(dtype=np.float32)

## 9) Moving Averages

In [49]:
import ta.trend


for _, price in enumerate(price_kinds):
    
    indicators[f'fe_ma_5_{price}'] = indicators[f'{price}'].rolling(window=5).mean().to_numpy(dtype=np.float32)
    indicators[f'fe_ma_10_{price}'] = indicators[f'{price}'].rolling(window=10).mean().to_numpy(dtype=np.float32)
    indicators[f'fe_ma_20_{price}'] = indicators[f'{price}'].rolling(window=20).mean().to_numpy(dtype=np.float32)
    indicators[f'fe_ma_50_{price}'] = indicators[f'{price}'].rolling(window=50).mean().to_numpy(dtype=np.float32)
    indicators[f'fe_ma_100_{price}'] = indicators[f'{price}'].rolling(window=100).mean().to_numpy(dtype=np.float32)

    indicators[f'fe_wma_5_{price}'] = ta.trend.WMAIndicator(indicators[f'{price}'], window=5).wma().to_numpy(dtype=np.float32)
    indicators[f'fe_wma_10_{price}'] = ta.trend.WMAIndicator(indicators[f'{price}'], window=10).wma().to_numpy(dtype=np.float32)
    indicators[f'fe_wma_20_{price}'] = ta.trend.WMAIndicator(indicators[f'{price}'], window=20).wma().to_numpy(dtype=np.float32)
    indicators[f'fe_wma_50_{price}'] = ta.trend.WMAIndicator(indicators[f'{price}'], window=50).wma().to_numpy(dtype=np.float32)
    indicators[f'fe_wma_100_{price}'] = ta.trend.WMAIndicator(indicators[f'{price}'], window=100).wma().to_numpy(dtype=np.float32)

    indicators[f'fe_ema_5_{price}'] = ta.trend.EMAIndicator(indicators[f'{price}'], window=5).ema_indicator().to_numpy(dtype=np.float32)
    indicators[f'fe_ema_10_{price}'] = ta.trend.EMAIndicator(indicators[f'{price}'], window=10).ema_indicator().to_numpy(dtype=np.float32)
    indicators[f'fe_ema_20_{price}'] = ta.trend.EMAIndicator(indicators[f'{price}'], window=20).ema_indicator().to_numpy(dtype=np.float32)
    indicators[f'fe_ema_50_{price}'] = ta.trend.EMAIndicator(indicators[f'{price}'], window=50).ema_indicator().to_numpy(dtype=np.float32)
    indicators[f'fe_ema_100_{price}'] = ta.trend.EMAIndicator(indicators[f'{price}'], window=100).ema_indicator().to_numpy(dtype=np.float32)

## 10) Ichimokou

In [53]:
ichimoku = ta.trend.IchimokuIndicator(high=indicators['high'], low=indicators['low'], window1=9, window2=26, window3=52)

# Add components to the DataFrame
indicators['fe_tenkan_sen'] = ichimoku.ichimoku_conversion_line().to_numpy(dtype=np.float32)
indicators['fe_kijun_sen'] = ichimoku.ichimoku_base_line().to_numpy(dtype=np.float32)
indicators['fe_senkou_span_a'] = ichimoku.ichimoku_a().to_numpy(dtype=np.float32)
indicators['fe_senkou_span_b'] = ichimoku.ichimoku_b().to_numpy(dtype=np.float32)
indicators['fe_chikou_span'] = indicators['close'].shift(-26).to_numpy(dtype=np.float32)


In [ ]:
indicators.columns

## 12) Denoised prices

In [61]:
from denoise.dwt import *
indicators['close_denoised'] = wavelet_denoising(indicators['close'])
indicators['open_denoised'] = wavelet_denoising(indicators['open'])
indicators['high_denoised'] = wavelet_denoising(indicators['high'])
indicators['low_denoised'] = wavelet_denoising(indicators['low'])


## 13) Binned Volume

In [67]:
num_bins = 11
labels = [ i+1 for i in range(num_bins)]
indicators['fe_volume_binned'] = pd.cut(indicators['volume'], bins=num_bins, labels=labels)

## 14) Momentum

In [ ]:
look_back_period = 5
indicators['fe_momentum_norm'] = indicators['close'].diff(look_back_period).to_numpy(dtype=np.float32)/indicators['close'].shift(look_back_period)
indicators['fe_momentum_norm_binned'] = pd.cut(indicators['fe_momentum_norm'], bins=num_bins, labels=labels)

## Williams

In [ ]:
indicators['fe_williams'] = ta.momentum.WilliamsRIndicator(high=indicators['high'], low=indicators['low'], close=indicators['close'], lbp=14).williams_r().to_numpy(dtype=np.float32)

# Save

In [74]:
indicators = indicators.astype(np.float32)

In [ ]:
os.getcwd()

In [79]:
indicators.to_csv('cnn/indicators.csv')

# Train Val Test split

In [ ]:
# indicators.columns

In [48]:
# indicators = indicators.drop(columns=['fe_adx', 'fe_adx_+DI', 'fe_adx_-DI',
#        'fe_aroon_up', 'fe_aroon_down', 'fe_so_hk', 'fe_so_hs'])

In [ ]:
print(f' indicators.shape *before* cleaning is:{indicators.shape}')
indicators = indicators.dropna()
print(f' indicators.shape *after* cleaning is:{indicators.shape}')


In [ ]:
train = indicators[:int(np.ceil(TRAIN_SPLIT*len(indicators)))]
val= indicators[int(np.ceil(TRAIN_SPLIT*len(indicators))): int(np.ceil( (TRAIN_SPLIT+VAL_SPLIT)*len(indicators) ))]
test= indicators[int(np.ceil((TRAIN_SPLIT+VAL_SPLIT)*len(indicators))):]
assert len(train) + len(val) + len(test) == len(indicators), f"The dimension of price is dividable to SPLIT numbers. Splited len indicators is:{len(train)+len(val)+len(test)} but df len is:{len(indicators)} "
print(f'train size:{train.shape[0]}, val size:{val.shape[0]}, test size:{test.shape[0]}.')

In [67]:
indicators.to_csv('cnn/indicators.csv', index=False)